Notebook to set up the first pipe line for a speech-to-speech translation. 

In [1]:
from transformers import pipeline
import torch 

d:\Code\Learning\audio-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.cuda.is_available()

True

In [3]:
classifier = pipeline(
    "audio-classification", model="MIT/ast-finetuned-speech-commands-v2", device=device
)

Loading weights: 100%|██████████| 203/203 [00:00<00:00, 12236.38it/s]


In [4]:
classifier.model.config.id2label

{0: 'backward',
 1: 'follow',
 2: 'five',
 3: 'bed',
 4: 'zero',
 5: 'on',
 6: 'learn',
 7: 'two',
 8: 'house',
 9: 'tree',
 10: 'dog',
 11: 'stop',
 12: 'seven',
 13: 'eight',
 14: 'down',
 15: 'six',
 16: 'forward',
 17: 'cat',
 18: 'right',
 19: 'visual',
 20: 'four',
 21: 'wow',
 22: 'no',
 23: 'nine',
 24: 'off',
 25: 'three',
 26: 'left',
 27: 'marvin',
 28: 'yes',
 29: 'up',
 30: 'sheila',
 31: 'happy',
 32: 'bird',
 33: 'go',
 34: 'one'}

In [5]:
from transformers.pipelines.audio_utils import ffmpeg_microphone_live


def launch_fn(
    wake_word="marvin",
    prob_threshold=0.5,
    chunk_length_s=2.0,
    stream_chunk_s=0.25,
    debug=False,
):
    if wake_word not in classifier.model.config.label2id.keys():
        raise ValueError(
            f"Wake word {wake_word} not in set of valid class labels, pick a wake word in the set {classifier.model.config.label2id.keys()}."
        )

    sampling_rate = classifier.feature_extractor.sampling_rate

    mic = ffmpeg_microphone_live(
        sampling_rate=sampling_rate,
        chunk_length_s=chunk_length_s,
        stream_chunk_s=stream_chunk_s,
    )


    for chunk in mic:
        # Ignore the 0.25 s temporary pieces.
        if chunk["partial"]:
            continue

        # This will be approximately 2 seconds at 16 kHz.
        print("Classifying chunk:", chunk["raw"].shape)

        prediction = classifier(chunk)[0]

        if debug:
            print(prediction)

        if (
            prediction["label"] == wake_word
            and prediction["score"] > prob_threshold
        ):
            print("Wake word detected.")
            return True

In [ ]:
launch_fn(debug=True)

Using microphone: Headset Microphone (2- DualSense Wireless Controller)
Classifying chunk: (32000,)
{'score': 0.08265265077352524, 'label': 'down'}
Classifying chunk: (32000,)
{'score': 0.08344399929046631, 'label': 'go'}
Classifying chunk: (32000,)
{'score': 0.07015644758939743, 'label': 'down'}
Classifying chunk: (32000,)
{'score': 0.18767990171909332, 'label': 'down'}
Classifying chunk: (32000,)
{'score': 0.10807015746831894, 'label': 'down'}


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Classifying chunk: (32000,)
{'score': 0.08842019736766815, 'label': 'nine'}
Classifying chunk: (32000,)
{'score': 0.12281502038240433, 'label': 'nine'}
Classifying chunk: (32000,)
{'score': 0.06701933592557907, 'label': 'down'}
Classifying chunk: (32000,)
{'score': 0.060357265174388885, 'label': 'down'}
Classifying chunk: (32000,)
{'score': 0.24323824048042297, 'label': 'marvin'}
Classifying chunk: (32000,)
{'score': 0.0733032152056694, 'label': 'down'}
Classifying chunk: (32000,)
{'score': 0.07772360742092133, 'label': 'bed'}
Classifying chunk: (32000,)
{'score': 0.06048876419663429, 'label': 'two'}
Classifying chunk: (32000,)
{'score': 0.062125369906425476, 'label': 'down'}
Classifying chunk: (32000,)
{'score': 0.6182573437690735, 'label': 'marvin'}
Wake word detected.


True

In [6]:
transcriber = pipeline(
    "automatic-speech-recognition", model="openai/whisper-small.en", device=device
)

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 11114.15it/s]


In [7]:
import sys

In [25]:
def transcribe_audio(chunk_length_s=3.0, stream_chunk_s=1.0):
    sampling_rate = transcriber.feature_extractor.sampling_rate

    mic = ffmpeg_microphone_live(
        sampling_rate=sampling_rate,
        chunk_length_s=chunk_length_s,
        stream_chunk_s=stream_chunk_s
    )

    print("Start speaking bsdk...")

    for chunk in mic: 
        if chunk["partial"]: 
            continue

        result = transcriber(chunk  )

        print(f"Transcription result bsdk: {result}")

        text = result["text"]

        sys.stdout.write("\r\033[K" + text)
        sys.stdout.flush()


   

In [23]:
# Undo the incompatible language/task configuration.
transcriber.model.generation_config.language = None
transcriber.model.generation_config.task = None

# Keep output bounded without passing generate_kwargs on every call.
transcriber.model.generation_config.max_new_tokens = 64

# to print ts just do transcribe_audio() lol 

In [38]:
from huggingface_hub import get_token
import requests

In [59]:
import os 

API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
}

def query(payload):
    if "messages" not in payload:
        payload= build_prompt(payload)
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()["choices"][0]["message"]["content"]

 

In [49]:
query("Will you dance with a stranger")

{'error': 'Unexpected token \'"\', ""Will you "... is not valid JSON'}

In [60]:
def build_prompt(user_message):
    return {
        "messages": [
            {
                "role": "user",
                "content": user_message
            }
        ],
        "model": "Qwen/Qwen2.5-32B-Instruct:featherless-ai"
    } 

In [61]:
query("Will you dance with a stranger")


"As an artificial intelligence, I don't have a physical form, so I can't actually dance or interact physically with people. However, I'm here to help answer your questions and provide information or entertainment in the ways I can! If you're interested in dancing or social interactions, I could certainly offer advice or information on those topics. What would you like to know?"